# Model Viewer App - One Click Install

Run the cell below to deploy the **Model Viewer** app to your workspace.  
It will create the app, deploy it, and print the shareable URL.

In [ ]:
APP_NAME = "model-viewer-app"

# ── One Click Install ─────────────────────────────────────────────────────────
import time, json, requests

ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
host, token = ctx.apiUrl().get(), ctx.apiToken().get()
H = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Resolve app source path relative to this notebook
nb = ctx.notebookPath().get()
src = "/".join(nb.split("/")[:-1]) + "/model-viewer-app"

# Verify source exists
r = requests.get(f"{host}/api/2.0/workspace/get-status", headers=H, params={"path": f"{src}/app.yaml"})
assert r.status_code == 200, f"App source not found at {src}. Ensure the repo is synced to your workspace."
print(f"[1/4] Source verified: {src}")

# Create or reuse app
r = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H)
if r.status_code == 200:
    print(f"[2/4] App '{APP_NAME}' exists, redeploying...")
else:
    print(f"[2/4] Creating app '{APP_NAME}'...")
    r = requests.post(f"{host}/api/2.0/apps", headers=H, json={"name": APP_NAME, "description": "Data Model Viewer - interactive graph visualization"})
    assert r.status_code in (200, 201), f"Failed to create app: {r.text}"
    print(f"       App created. Waiting for compute to start...")
    # New apps need compute to spin up before first deploy
    for _ in range(30):
        time.sleep(10)
        cs = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json().get("compute_status", {}).get("state", "")
        print(f"       Compute: {cs}")
        if cs == "ACTIVE":
            break
    else:
        raise Exception("Compute did not start within 5 minutes.")

# Deploy
print(f"[3/4] Deploying...")
r = requests.post(f"{host}/api/2.0/apps/{APP_NAME}/deployments", headers=H, json={"source_code_path": src, "mode": "SNAPSHOT"})
assert r.status_code in (200, 201), f"Deploy failed: {r.text}"

# Wait for deployment
for i in range(30):
    time.sleep(10)
    info = requests.get(f"{host}/api/2.0/apps/{APP_NAME}", headers=H).json()
    state = info.get("active_deployment", {}).get("status", {}).get("state", "")
    msg = info.get("active_deployment", {}).get("status", {}).get("message", "")
    if state == "SUCCEEDED":
        url = info["url"]
        print(f"[4/4] Deployed!")
        print(f"\n{'='*60}")
        print(f"  App URL: {url}")
        print(f"{'='*60}")
        displayHTML(f'<h2 style="color:#4ECDC4">Model Viewer Deployed!</h2><p style="font-size:16px"><a href="{url}" target="_blank">{url}</a></p><p>Upload a <code>model.json</code> to visualize your data model.</p>')
        break
    elif state == "FAILED":
        raise Exception(f"Deployment failed: {msg}")
    print(f"       {state}: {msg}")
else:
    print("Timed out. Check app status in the workspace UI.")